In [96]:
# =========================
# 1. Install dependencies
# =========================
# pip install pandas numpy scikit-learn torch transformers openai tqdm

# =========================
# 2. Imports + config
# =========================
import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from openai import OpenAI

DATA_PATH = "datset.csv"   # change to "dataset.csv" if needed
OPENAI_MODEL = "gpt-5-mini"   # lower-cost LLM for feature engineering
DISTILBERT_MODEL = "distilbert-base-uncased"
MAX_LEN = 256
LOOKBACK_TXNS = 10
USE_LLM = True
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
import os


client = OpenAI()


In [97]:
DATA_PATH="../../transactions/card_transaction.v1.csv"
df = pd.read_csv(DATA_PATH)

In [125]:


# ================================
# 🔥 CLEAN DATA (MANDATORY)
# ================================

# 1. Clean Amount ($ → float)
df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

# 2. Convert Time → Hour (0–23)
df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour

# If Time is numeric seconds instead:
# df["Hour"] = (df["Time"] // 3600) % 24

# 3. Convert Use Chip → 0/1
df["Use Chip"] = df["Use Chip"].map({"Yes": 1, "No": 0}).fillna(0)

# 4. Convert Errors → 0/1
df["Errors?"] = (df["Errors?"] != "None").astype(int)

# 5. Convert Zip → numeric
df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")
# Try numeric first
df["Time"] = pd.to_numeric(df["Time"], errors="coerce")

# Convert seconds → hour
df["Hour"] = (df["Time"] // 3600) % 24
df["Is Fraud?"] = df["Is Fraud?"].map({"Yes": 1, "No": 0})
# 6. Fill missing values
df = df.fillna(0)

# ================================
# 🔍 VERIFY (VERY IMPORTANT)
# ================================
print(df.dtypes)
print(df.head())

/tmp/ipykernel_3335815/728849075.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour


User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time              float64
Amount            float64
Use Chip          float64
Merchant Name       int64
Merchant City      object
Merchant State     object
Zip               float64
MCC                 int64
Errors?             int64
Is Fraud?           int64
Hour              float64
dtype: object
   User  Card  Year  Month  Day  Time  Amount  Use Chip        Merchant Name  \
0     0     0  2002      9    1   0.0  134.09       0.0  3527213246127876953   
1     0     0  2002      9    1   0.0   38.48       0.0  -727612092139916043   
2     0     0  2002      9    2   0.0  120.34       0.0  -727612092139916043   
3     0     0  2002      9    2   0.0  128.95       0.0  3414527459579106770   
4     0     0  2002      9    3   0.0  104.71       0.0  5817218446178736267   

   Merchant City Merchant State      Zip   MCC  Errors?  Is Fraud?  Hour  
0       La Ver

In [126]:
df["Merchant City"] = df["Merchant City"].astype(str)
df["Merchant State"] = df["Merchant State"].astype(str)

In [127]:
print(df.dtypes)
print(df.head())

User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time              float64
Amount            float64
Use Chip          float64
Merchant Name       int64
Merchant City      object
Merchant State     object
Zip               float64
MCC                 int64
Errors?             int64
Is Fraud?           int64
Hour              float64
dtype: object
   User  Card  Year  Month  Day  Time  Amount  Use Chip        Merchant Name  \
0     0     0  2002      9    1   0.0  134.09       0.0  3527213246127876953   
1     0     0  2002      9    1   0.0   38.48       0.0  -727612092139916043   
2     0     0  2002      9    2   0.0  120.34       0.0  -727612092139916043   
3     0     0  2002      9    2   0.0  128.95       0.0  3414527459579106770   
4     0     0  2002      9    3   0.0  104.71       0.0  5817218446178736267   

   Merchant City Merchant State      Zip   MCC  Errors?  Is Fraud?  Hour  
0       La Ver

In [98]:
# 🔥 Model

from dotenv import load_dotenv
import os
import google.generativeai as genai

load_dotenv()

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

In [99]:
PROMPT = """
You are a fraud analytics feature engineering assistant.

The features you generate will be used to train a machine learning model:
RandomForestClassifier from sklearn.

Dataset columns:
['User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount', 'Use Chip',
 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC',
 'Errors?', 'Is Fraud?']

Your task is to design GLOBAL USER-LEVEL FEATURES.

STRICT REQUIREMENTS (VERY IMPORTANT):

- Features must be NUMERIC (int or float only)
- No strings, no lists, no text outputs
- Must be directly usable in sklearn models (no preprocessing required)
- Must be computed using pandas groupby('User')
- Same features for ALL users (global features)

Allowed operations:
- mean, std, min, max, sum
- count, nunique
- ratios (division of aggregates)
- boolean → numeric (0/1)

Avoid:
- raw text columns (Merchant Name, Errors?, City, etc.)
- concatenated strings
- per-user custom logic

Output STRICTLY in JSON:

[
  {
    "feature_name": "...",
    "pandas_code": "...",
    "description": "...",
    "type": "numeric"
  }
]

Generate 12–15 high-quality features optimized for fraud detection using RandomForest.

Focus on:
- spending behavior
- transaction frequency
- merchant diversity
- geographic spread
- time patterns
- error ratios
- chip usage behavior

Do NOT compute values.
Do NOT predict fraud.
Do NOT output anything outside JSON.
"""

In [100]:
# for m in genai.list_models():
#     print(m.name, "->", m.supported_generation_methods)

In [101]:
model = genai.GenerativeModel("models/gemini-flash-lite-latest")


In [102]:
# ================================
# 🔥 FORCE CLEAN RAW DATA
# ================================

# Amount → remove $ and convert
df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

# Time → convert to hour
df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour

# If Time is numeric already (seconds), use:
# df["Hour"] = (df["Time"] // 3600) % 24

# Use Chip → 0/1
df["Use Chip"] = df["Use Chip"].map({"Yes": 1, "No": 0}).fillna(0)

# Errors → 0/1
df["Errors?"] = (df["Errors?"] != "None").astype(int)

# Zip → numeric
df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")

# Fill missing
df = df.fillna(0)

/tmp/ipykernel_3335815/2132959710.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour


In [103]:
response = model.generate_content(PROMPT)

output = response.text
# print("Raw Output:\n", output)

output = output.replace("```json", "").replace("```", "").strip()

features = json.loads(output)

# =========================================
# ✅ APPLY FEATURES USING PANDAS
# =========================================

# Example: load your dataset
# df = pd.read_csv("your_dataset.csv")

user_df = pd.DataFrame()

for f in features:
    try:
        print("Applying:", f["feature_name"])
        user_df[f["feature_name"]] = eval(f["pandas_code"])
    except Exception as e:
        print("Error in", f["feature_name"], ":", e)

# ✅ Add label
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

# ✅ Fill missing
user_df = user_df.fillna(0)

print("\nFinal Shape:", user_df.shape)
print(user_df.head())

Applying: user_avg_amount
Applying: user_std_amount
Applying: user_txn_count
Applying: user_avg_time_of_day_sin
Error in user_avg_time_of_day_sin : unsupported operand type(s) for /: 'float' and 'str'
Applying: user_unique_merchants_ratio
Applying: user_avg_chip_usage
Applying: user_max_daily_txns
Applying: user_unique_zip_count
Applying: user_avg_amount_last_month
Applying: user_error_ratio
Applying: user_avg_amount_vs_global_mean_ratio
Applying: user_relative_txn_frequency_7d
Error in user_relative_txn_frequency_7d : 'User'
Applying: user_time_since_last_txn_std
Error in user_time_since_last_txn_std : unsupported operand type(s) for -: 'str' and 'str'

Final Shape: (2000, 11)
      user_avg_amount  user_std_amount  user_txn_count  \
User                                                     
0           81.299989        94.159093           19963   
1           81.118050       156.784691            8919   
2           35.159687        54.298552           41978   
3          117.277603  

In [104]:
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

In [105]:
# Separate label first
y = user_df["is_fraud_user"]

# Remove non-numeric features
X = user_df.drop(columns=["is_fraud_user"])
X = X.select_dtypes(include=["number"])

In [106]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # IMPORTANT for fraud datasets
)

In [107]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"   # IMPORTANT for fraud
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [108]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

          No       0.89      0.71      0.79       131
         Yes       0.87      0.96      0.91       269

    accuracy                           0.88       400
   macro avg       0.88      0.83      0.85       400
weighted avg       0.88      0.88      0.87       400

ROC-AUC: 0.8954283606231732


In [109]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print(importance.head(10))

user_txn_count                          0.279151
user_unique_zip_count                   0.224203
user_unique_merchants_ratio             0.203386
user_avg_amount_last_month              0.104110
user_max_daily_txns                     0.058597
user_std_amount                         0.051622
user_avg_amount                         0.039950
user_avg_amount_vs_global_mean_ratio    0.038980
user_avg_chip_usage                     0.000000
user_error_ratio                        0.000000
dtype: float64


In [110]:
feature_names = [f["feature_name"] for f in features]

In [111]:
top_features = importance.head(10).to_dict()

In [112]:
report1 = classification_report(y_test, y_pred)
roc1 = roc_auc_score(y_test, y_prob)

llm_input = {
    "existing_features": feature_names,
    "classification_report": report1,
    "roc_auc": roc1,
    "top_features": top_features
}

In [113]:
PROMPT = f"""
You are a fraud detection ML expert.

A RandomForest model was trained using the following USER-LEVEL features:

{llm_input["existing_features"]}

Model Performance:

Classification Report:
{llm_input["classification_report"]}

ROC-AUC:
{llm_input["roc_auc"]}

Top Important Features:
{llm_input["top_features"]}

Your task:

1. Analyze weaknesses (especially fraud recall)
2. Identify missing behavioral signals
3. Suggest 8–12 NEW or IMPROVED features

STRICT RULES:
- Features must be numeric
- Must be computable using pandas groupby('User')
- Must NOT duplicate existing features
- Must improve fraud detection

Output STRICTLY in JSON:

[
  {{
    "feature_name": "...",
    "pandas_code": "...",
    "reason": "..."
  }}
]
"""

In [114]:
model_llm = genai.GenerativeModel("models/gemini-flash-lite-latest")


In [115]:
response = model_llm.generate_content(PROMPT)

output = response.text.replace("```json", "").replace("```", "").strip()

new_features = json.loads(output)

In [116]:
base_features = []

for f in features:
    code = f["pandas_code"]
    
    # Only keep df-based features
    if "df.groupby" in code and "user_" not in code.split("=")[-1]:
        base_features.append(f)

In [118]:
# # Amount → float
# df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

# # Time → hour (0–23)
# df["Hour"] = pd.to_datetime(df["Time"], format="%H:%M:%S", errors="coerce").dt.hour

# # Use Chip → 0/1
# df["Use Chip"] = (df["Use Chip"] == "Yes").astype(int)

# # Errors → 0/1
# df["Errors?"] = (df["Errors?"] != "None").astype(int)

# # Zip → numeric
# df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")

# # Fill missing
# df = df.fillna(0)

In [119]:
for f in base_features:
    try:
        user_df[f["feature_name"]] = eval(f["pandas_code"])
    except Exception as e:
        print("Base error:", f["feature_name"], e)

Base error: user_avg_time_of_day_sin unsupported operand type(s) for /: 'float' and 'str'
Base error: user_relative_txn_frequency_7d 'User'
Base error: user_time_since_last_txn_std unsupported operand type(s) for -: 'str' and 'str'


In [120]:
X = user_df.drop(columns=["is_fraud_user"])
y = user_df["is_fraud_user"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [121]:
# ================================
# 🔹 9. TRAIN MODEL
# ================================
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# ================================
# 🔹 10. PREDICT
# ================================
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# ================================
# 🔹 11. EVALUATE
# ================================
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("\n=== ROC AUC ===")
print(roc_auc_score(y_test, y_prob))

# ================================
# 🔹 12. FEATURE IMPORTANCE
# ================================
importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("\n=== Top Features ===")
print(importance.head(10))


=== Classification Report ===
              precision    recall  f1-score   support

          No       0.88      0.69      0.77       131
         Yes       0.86      0.95      0.91       269

    accuracy                           0.87       400
   macro avg       0.87      0.82      0.84       400
weighted avg       0.87      0.87      0.86       400


=== ROC AUC ===
0.8986066573966345

=== Top Features ===
user_txn_count                          0.272551
user_unique_zip_count                   0.210721
user_unique_merchants_ratio             0.209214
user_avg_amount_last_month              0.101575
user_std_amount                         0.059142
user_max_daily_txns                     0.057658
user_avg_amount                         0.045563
user_avg_amount_vs_global_mean_ratio    0.043576
user_avg_chip_usage                     0.000000
user_error_ratio                        0.000000
dtype: float64


In [122]:
report2 = classification_report(y_test, y_pred)
roc2 = roc_auc_score(y_test, y_prob)

In [123]:
# precision    recall  f1-score   support

#           No       0.87      0.72      0.79       131
#          Yes       0.87      0.95      0.91       269

#     accuracy                           0.87       400
#    macro avg       0.87      0.83      0.85       400
# weighted avg       0.87      0.87      0.87       400

# ROC-AUC: 0.9174210391895343